# 07. Model comparison

Compares the baselines of notebook 06 with random forest, XGBoost and LightGBM on the **same rows and the same walk-forward protocol** (expanding window, rolling-origin blocks by season from winter 2020-21, a gap equal to the lead between training and test).

Two test views, as in notebook 06:

- **observed**: weather models receive the weather that happened on the target day in the forecast columns (a perfect forecast, so an **upper bound**). Covers all winters 2020-21 to 2024-25.
- **forecast**: weather models receive the real archived forecast (only from 2024, about 140 winter days). This is the realistic view and the one to quote for headline numbers, with its small sample in mind.

Errors are in ug/m3. Intervals are 95% paired block-bootstrap intervals (14-day blocks).

In [1]:
import warnings
from datetime import date
from pathlib import Path

import pandas as pd

from aeron.evaluation import report, splits
from aeron.evaluation.runner import common_rows, run_models
from aeron.models import ablations, baselines, trees

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

daily = pd.read_csv(ROOT / "data" / "processed" / "daily_us_embassy.csv", index_col=0, parse_dates=True)
blocks = splits.seasonal_blocks(2020, date(2025, 2, 25))
LEADS = (1, 2, 3)
VIEWS = ("observed", "forecast")
BASELINES = ["persistence", "climatology", "ridge_history", "ridge_forecast_weather", "ridge_forecast_no_wind"]
TREES = [m.name for m in trees.random_forest_models() + trees.boosted_models()]
MAIN = BASELINES + TREES


def main_models():
    base = [m for m in baselines.default_models() if m.name in BASELINES]
    return base + trees.random_forest_models() + trees.boosted_models()


results, fitted = {}, {}
for view in VIEWS:
    for lead in LEADS:
        models = main_models()
        results[(view, lead)] = run_models(daily, models, lead, blocks, view)
        fitted[(view, lead)] = models
print({(v, l): len(common_rows(results[(v, l)], MAIN)) for v in VIEWS for l in LEADS})

{('observed', 1): 1172, ('observed', 2): 1144, ('observed', 3): 1129, ('forecast', 1): 363, ('forecast', 2): 358, ('forecast', 3): 354}


## MLflow logging

In [2]:
try:
    from aeron.evaluation.tracking import tracked_run
except ImportError:
    tracked_run = None
    print("MLflow not installed, runs are not logged")


def log_runs(results, fitted, names, prefix, task):
    logged = 0
    for (view, lead), pred in results.items():
        rows = common_rows(pred, names)
        summary = report.summarize(rows, names)
        for model in fitted[(view, lead)]:
            if model.name not in names:
                continue
            sub = summary[summary["model"] == model.name]
            params = {
                "model": model.name,
                "lead_days": lead,
                "test_view": view,
                "threshold": report.EVENT_THRESHOLD,
                "alpha": getattr(model, "chosen_alpha", None),
                "settings": getattr(model, "chosen", None),
                "wind_ratio": getattr(model, "wind_ratio", None),
            }
            split = {
                "scheme": "walk-forward, expanding window",
                "first_block": blocks[0].name,
                "last_block": blocks[-1].name,
                "gap_days": lead,
            }
            with tracked_run(
                f"{prefix}-{view}-lead{lead}-{model.name}", params, seed=0, split=split,
                features=list(getattr(model, "_columns", [])), tags={"task": task},
            ) as run:
                for _, row in sub.iterrows():
                    run.log_metrics(
                        {k: row[k] for k in ("n", "mae", "rmse", "bias", "r2", "precision", "recall", "f1", "always_yes_f1")},
                        prefix=row["group"],
                    )
                run.log_predictions(rows[["target_date", "block", "season", "y", model.name]])
            logged += 1
    return logged


if tracked_run is not None:
    print("logged", log_runs(results, fitted, MAIN, "compare", "3.4 model comparison"), "runs")

logged 84 runs


## Winter MAE and RMSE, observed-weather view (upper bound)

In [3]:
def mae_table(view, group, names=MAIN):
    cols = {}
    for lead in LEADS:
        s = report.summarize(common_rows(results[(view, lead)], names), names)
        s = s[s["group"] == group].set_index("model")
        cols[(f"lead {lead}", "MAE")] = s["mae"]
        cols[(f"lead {lead}", "RMSE")] = s["rmse"]
        cols[(f"lead {lead}", "bias")] = s["bias"]
        cols[(f"lead {lead}", "n")] = s["n"]
    return pd.DataFrame(cols).round(1)


mae_table("observed", "winter")

lead 1                 lead 2                 lead 3                 
                          MAE  RMSE bias    n    MAE  RMSE bias    n    MAE  RMSE  bias    n
model                                                                                       
persistence              21.7  30.6  1.3  467   28.5  42.5  0.8  456   30.7  44.9   0.4  446
climatology              26.8  37.3 -8.0  467   27.1  38.9 -7.9  456   27.2  39.4  -7.8  446
ridge_history            17.6  24.1 -2.0  467   23.5  34.3 -3.3  456   25.1  36.0  -4.8  446
ridge_forecast_weather   15.7  21.6 -2.4  467   19.7  30.3 -3.8  456   20.6  31.6  -4.9  446
ridge_forecast_no_wind   16.2  22.4 -2.5  467   20.8  31.4 -4.2  456   21.8  32.8  -5.4  446
rf_history               17.7  24.4 -3.1  467   23.9  34.5 -4.2  456   25.4  36.5  -5.0  446
rf_forecast_weather      15.1  21.1 -2.0  467   18.5  28.8 -2.9  456   19.1  30.2  -3.8  446
rf_forecast_no_wind      16.4  22.9 -2.8  467   20.7  31.1 -4.8  456   21.8  33.1  -5.1  446
xgb_history              18.8  26.4 -6.3  467   24.7  36.3 -8.4  456   26.3  38.5 -10.3  446
xgb_forecast_weather     15.1  21.7 -4.1  467   18.9  30.0 -5.9  456   18.4  30.4  -7.5  446
xgb_forecast_no_wind     16.9  24.2 -5.2  467   20.4  31.8 -7.6  456   21.0  33.2  -9.1  446
lgbm_history             18.7  26.5 -7.0  467   24.5  36.3 -8.8  456   26.1  38.8 -10.2  446
lgbm_forecast_weather    15.5  22.5 -5.4  467   18.6  29.9 -7.2  456   19.2  31.0  -8.5  446
lgbm_forecast_no_wind    16.7  23.8 -6.0  467   21.0  32.3 -7.8  456   21.2  33.4 -10.6  446

## Winter MAE and RMSE, real archived forecasts (realistic view, small sample)

In [4]:
mae_table("forecast", "winter")

lead 1                 lead 2                 lead 3                 
                          MAE  RMSE bias    n    MAE  RMSE bias    n    MAE  RMSE  bias    n
model                                                                                       
persistence              24.1  31.0  0.2  146   32.4  41.6 -0.1  143   35.1  44.5  -0.3  140
climatology              25.9  32.1 -6.7  146   25.7  32.0 -6.9  143   25.8  32.1  -7.1  140
ridge_history            19.0  24.3 -0.4  146   25.8  32.1 -2.9  143   27.0  33.4  -3.3  140
ridge_forecast_weather   18.2  23.3  3.9  146   22.4  28.1  2.2  143   23.6  28.9   1.6  140
ridge_forecast_no_wind   17.5  22.9 -0.8  146   22.5  28.6 -4.6  143   23.6  29.7  -5.3  140
rf_history               19.4  24.9 -0.9  146   26.0  32.1 -1.7  143   26.7  32.8  -1.5  140
rf_forecast_weather      18.0  23.6  6.6  146   22.9  28.1  9.0  143   23.5  28.9  10.4  140
rf_forecast_no_wind      17.2  22.8 -0.3  146   21.9  27.8 -2.6  143   22.4  27.8  -2.7  140
xgb_history              19.7  25.7 -6.2  146   27.4  34.2 -6.6  143   27.4  34.9  -8.8  140
xgb_forecast_weather     17.7  23.1  5.4  146   22.1  27.7  6.1  143   22.0  28.0   7.9  140
xgb_forecast_no_wind     17.3  23.1 -4.9  146   21.0  27.3 -7.3  143   21.8  28.4  -8.1  140
lgbm_history             19.0  25.3 -5.3  146   26.4  33.2 -7.0  143   26.3  34.0  -7.9  140
lgbm_forecast_weather    17.1  22.8  2.9  146   20.9  26.4  2.6  143   22.0  28.1   5.1  140
lgbm_forecast_no_wind    16.8  22.2 -4.9  146   21.6  27.9 -7.0  143   21.4  28.0  -9.6  140

## March to October, observed view

In [5]:
mae_table("observed", "other")

lead 1                 lead 2                 lead 3                
                          MAE  RMSE bias    n    MAE  RMSE bias    n    MAE  RMSE bias    n
model                                                                                      
persistence               7.3  11.3 -0.4  705    9.2  13.9 -0.4  688   10.4  16.4  0.1  683
climatology              12.3  16.2  6.3  705   12.4  16.2  6.5  688   12.4  16.2  6.6  683
ridge_history             6.5   9.8  0.1  705    8.3  11.9  1.3  688    8.9  12.6  2.0  683
ridge_forecast_weather    6.8   9.9  0.1  705    8.9  12.4  1.3  688    9.7  13.1  2.8  683
ridge_forecast_no_wind    6.8   9.8  0.6  705    9.0  12.4  2.4  688    9.7  13.3  3.1  683
rf_history                6.9  10.0  2.0  705    9.4  12.8  3.9  688    9.5  13.2  3.8  683
rf_forecast_weather       6.7   9.6  1.9  705    8.7  12.2  3.5  688    9.0  12.7  4.2  683
rf_forecast_no_wind       6.9   9.8  1.9  705    9.0  12.3  3.7  688    9.4  12.8  4.3  683
xgb_history               6.7   9.8  0.8  705    9.2  13.0  2.3  688    9.3  13.1  2.3  683
xgb_forecast_weather      6.3   9.4  0.8  705    7.8  11.8  1.7  688    8.4  12.3  2.6  683
xgb_forecast_no_wind      6.7   9.8  0.8  705    8.4  12.0  1.9  688    8.7  12.4  2.4  683
lgbm_history              6.9  10.1  0.8  705    8.7  12.8  1.5  688    9.1  13.0  1.8  683
lgbm_forecast_weather     6.1   9.2  0.1  705    7.9  11.4  2.0  688    8.0  12.0  1.7  683
lgbm_forecast_no_wind     6.5   9.7  0.7  705    8.1  11.9  1.8  688    8.7  12.5  2.4  683

## March to October 2024, real forecasts

In [6]:
mae_table("forecast", "other")

lead 1                  lead 2                  lead 3                 
                          MAE  RMSE  bias    n    MAE  RMSE  bias    n    MAE  RMSE  bias    n
model                                                                                         
persistence               8.5  14.7  -0.4  217   10.0  17.1  -0.3  215   11.5  21.0   0.6  214
climatology              16.4  21.1  10.6  217   16.4  21.1  11.1  215   16.0  20.3  11.6  214
ridge_history             7.7  12.9   0.9  217    9.6  15.3   2.2  215   10.1  15.6   3.7  214
ridge_forecast_weather   10.5  14.1   6.2  217   13.6  17.7   9.2  215   13.8  18.0  10.0  214
ridge_forecast_no_wind    8.0  12.9   0.5  217    9.6  15.4   1.7  215   10.0  15.7   3.2  214
rf_history                9.2  13.2   3.9  217   12.1  16.8   6.3  215   11.6  16.8   6.0  214
rf_forecast_weather      10.2  14.0   6.0  217   14.0  18.9  10.0  215   14.9  19.9  11.7  214
rf_forecast_no_wind       9.1  13.1   3.8  217   11.3  16.1   5.6  215   11.6  16.4   6.4  214
xgb_history               8.6  12.8   3.0  217   12.3  17.4   5.5  215   11.7  17.0   5.7  214
xgb_forecast_weather     10.2  14.4   5.8  217   13.3  18.7   8.6  215   13.6  19.1   9.7  214
xgb_forecast_no_wind      8.8  13.6   2.8  217   10.7  16.3   3.9  215   10.8  16.2   5.0  214
lgbm_history              8.8  13.4   3.0  217   11.5  17.2   4.3  215   11.5  16.9   5.4  214
lgbm_forecast_weather     9.9  13.7   5.5  217   11.8  16.8   6.8  215   12.5  18.0   8.2  214
lgbm_forecast_no_wind     8.4  13.3   2.5  217   10.3  16.1   3.8  215   10.6  16.4   4.7  214

## Paired differences in winter MAE

Negative means the first model has the lower error. Intervals that include 0 mean the difference is not distinguishable from noise with this sample.

In [7]:
PAIRS = [
    ("ridge_forecast_weather", "persistence"),
    ("ridge_forecast_weather", "climatology"),
    ("rf_forecast_weather", "ridge_forecast_weather"),
    ("xgb_forecast_weather", "ridge_forecast_weather"),
    ("lgbm_forecast_weather", "ridge_forecast_weather"),
    ("lgbm_forecast_weather", "rf_forecast_weather"),
    ("lgbm_forecast_weather", "xgb_forecast_weather"),
    ("lgbm_forecast_weather", "lgbm_history"),
    ("lgbm_forecast_no_wind", "lgbm_forecast_weather"),
]


def difference_table(view, group="winter"):
    rows = []
    for a, b in PAIRS:
        row = {"comparison": f"{a} minus {b}"}
        for lead in LEADS:
            d = report.paired_difference(common_rows(results[(view, lead)], MAIN), a, b, group)
            row[f"lead {lead}"] = f"{d['diff']:+.1f} [{d['lo']:+.1f}, {d['hi']:+.1f}]"
        rows.append(row)
    return pd.DataFrame(rows).set_index("comparison")


print("observed view")
display(difference_table("observed"))
print("real forecast view")
display(difference_table("forecast"))

observed view


,lead 1,lead 2,lead 3
comparison,,,
ridge_forecast_weather minus persistence,"-6.1 [-7.5, -4.7]","-8.8 [-11.3, -6.4]","-10.1 [-12.9, -7.4]"
ridge_forecast_weather minus climatology,"-11.2 [-14.1, -8.6]","-7.4 [-9.4, -5.8]","-6.6 [-8.5, -4.9]"
rf_forecast_weather minus ridge_forecast_weather,"-0.5 [-1.1, +0.0]","-1.2 [-2.2, -0.2]","-1.6 [-2.5, -0.5]"
xgb_forecast_weather minus ridge_forecast_weather,"-0.5 [-1.3, +0.3]","-0.7 [-2.1, +0.8]","-2.2 [-3.6, -0.7]"
lgbm_forecast_weather minus ridge_forecast_weather,"-0.2 [-1.2, +0.8]","-1.0 [-2.3, +0.3]","-1.4 [-2.9, +0.1]"
lgbm_forecast_weather minus rf_forecast_weather,"+0.3 [-0.5, +1.2]","+0.1 [-0.8, +1.1]","+0.1 [-0.9, +1.2]"
lgbm_forecast_weather minus xgb_forecast_weather,"+0.3 [-0.2, +0.9]","-0.3 [-0.8, +0.3]","+0.8 [+0.3, +1.2]"
lgbm_forecast_weather minus lgbm_history,"-3.2 [-4.2, -2.2]","-5.9 [-7.7, -4.0]","-6.9 [-9.3, -4.5]"
lgbm_forecast_no_wind minus lgbm_forecast_weather,"+1.3 [+0.6, +1.9]","+2.3 [+1.2, +3.5]","+2.1 [+1.0, +3.0]"


real forecast view


,lead 1,lead 2,lead 3
comparison,,,
ridge_forecast_weather minus persistence,"-6.0 [-7.7, -4.1]","-10.0 [-12.9, -7.3]","-11.5 [-15.7, -7.5]"
ridge_forecast_weather minus climatology,"-7.8 [-9.6, -5.8]","-3.3 [-5.0, -1.8]","-2.2 [-4.3, -0.2]"
rf_forecast_weather minus ridge_forecast_weather,"-0.1 [-0.9, +0.6]","+0.5 [-1.5, +2.5]","-0.1 [-2.1, +2.1]"
xgb_forecast_weather minus ridge_forecast_weather,"-0.5 [-1.5, +0.5]","-0.3 [-2.8, +2.4]","-1.6 [-3.9, +0.9]"
lgbm_forecast_weather minus ridge_forecast_weather,"-1.0 [-2.4, +0.3]","-1.5 [-3.7, +1.0]","-1.6 [-4.1, +1.2]"
lgbm_forecast_weather minus rf_forecast_weather,"-0.9 [-2.3, +0.4]","-1.9 [-3.2, -0.7]","-1.5 [-3.5, +0.7]"
lgbm_forecast_weather minus xgb_forecast_weather,"-0.6 [-1.9, +0.7]","-1.1 [-2.1, -0.1]","+0.0 [-1.2, +1.3]"
lgbm_forecast_weather minus lgbm_history,"-1.9 [-3.9, -0.1]","-5.4 [-8.3, -2.5]","-4.3 [-8.6, +0.4]"
lgbm_forecast_no_wind minus lgbm_forecast_weather,"-0.3 [-1.4, +0.9]","+0.6 [-1.2, +2.4]","-0.6 [-2.5, +1.4]"


## MAE per test block (lead 1 and 3, observed view)

In [8]:
show = ["persistence", "climatology", "ridge_forecast_weather", "rf_forecast_weather", "xgb_forecast_weather", "lgbm_forecast_weather"]
for lead in (1, 3):
    print("lead", lead)
    print(report.per_block_mae(common_rows(results[("observed", lead)], MAIN), show).round(1).to_string())
    print()

lead 1
                days  persistence  climatology  ridge_forecast_weather  rf_forecast_weather  xgb_forecast_weather  lgbm_forecast_weather
block                                                                                                                                   
winter 2020-21    88         13.9         19.7                    10.9                 10.8                  11.9                   12.6
mar-oct 2021     176          7.2          8.3                     6.5                  6.1                   5.7                    5.5
winter 2021-22   102         18.0         18.0                    11.2                 11.6                  10.8                   11.4
mar-oct 2022     104          5.8          7.1                     6.1                  4.9                   4.7                    4.6
winter 2022-23    76         21.4         40.5                    18.1                 16.1                  16.5                   16.6
mar-oct 2023     208          6.8 

## Exceedance of 55.5 ug/m3 (daily mean), winter

In [9]:
def event_table(view, group="winter", names=MAIN):
    cols = {}
    for lead in LEADS:
        s = report.summarize(common_rows(results[(view, lead)], names), names)
        s = s[s["group"] == group].set_index("model")
        cols[(f"lead {lead}", "F1")] = s["f1"]
        cols[(f"lead {lead}", "recall")] = s["recall"]
        cols[(f"lead {lead}", "precision")] = s["precision"]
        cols[(f"lead {lead}", "always-yes F1")] = s["always_yes_f1"]
        cols[(f"lead {lead}", "base rate")] = s["base_rate"]
    return pd.DataFrame(cols).round(2)


print("observed view")
display(event_table("observed"))
print("real forecast view")
display(event_table("forecast"))

observed view


lead 1                                          lead 2                                          lead 3                                         
                           F1 recall precision always-yes F1 base rate     F1 recall precision always-yes F1 base rate     F1 recall precision always-yes F1 base rate
model                                                                                                                                                                 
persistence              0.70   0.70      0.70          0.62      0.45   0.60   0.60      0.61          0.62      0.45   0.58   0.57      0.58          0.62      0.45
climatology              0.45   0.40      0.52          0.62      0.45   0.46   0.40      0.54          0.62      0.45   0.46   0.41      0.54          0.62      0.45
ridge_history            0.73   0.72      0.74          0.62      0.45   0.62   0.61      0.64          0.62      0.45   0.53   0.49      0.58          0.62      0.45
ridge_forecast_weather   0.77   0.77      0.77          0.62      0.45   0.75   0.79      0.71          0.62      0.45   0.72   0.75      0.70          0.62      0.45
ridge_forecast_no_wind   0.77   0.78      0.77          0.62      0.45   0.74   0.77      0.71          0.62      0.45   0.70   0.71      0.69          0.62      0.45
rf_history               0.74   0.74      0.75          0.62      0.45   0.64   0.64      0.65          0.62      0.45   0.61   0.62      0.60          0.62      0.45
rf_forecast_weather      0.80   0.81      0.79          0.62      0.45   0.75   0.75      0.74          0.62      0.45   0.77   0.77      0.78          0.62      0.45
rf_forecast_no_wind      0.77   0.76      0.77          0.62      0.45   0.68   0.67      0.71          0.62      0.45   0.67   0.65      0.69          0.62      0.45
xgb_history              0.69   0.64      0.76          0.62      0.45   0.53   0.46      0.64          0.62      0.45   0.45   0.35      0.61          0.62      0.45
xgb_forecast_weather     0.79   0.78      0.80          0.62      0.45   0.72   0.67      0.78          0.62      0.45   0.71   0.66      0.78          0.62      0.45
xgb_forecast_no_wind     0.75   0.73      0.77          0.62      0.45   0.69   0.66      0.73          0.62      0.45   0.64   0.56      0.76          0.62      0.45
lgbm_history             0.71   0.65      0.77          0.62      0.45   0.56   0.48      0.68          0.62      0.45   0.48   0.38      0.64          0.62      0.45
lgbm_forecast_weather    0.79   0.77      0.82          0.62      0.45   0.71   0.66      0.78          0.62      0.45   0.70   0.63      0.79          0.62      0.45
lgbm_forecast_no_wind    0.77   0.74      0.79          0.62      0.45   0.68   0.63      0.74          0.62      0.45   0.62   0.52      0.78          0.62      0.45

real forecast view


lead 1                                          lead 2                                          lead 3                                         
                           F1 recall precision always-yes F1 base rate     F1 recall precision always-yes F1 base rate     F1 recall precision always-yes F1 base rate
model                                                                                                                                                                 
persistence              0.66   0.67      0.66          0.62      0.45   0.53   0.52      0.54          0.62      0.45   0.49   0.48      0.50          0.63      0.46
climatology              0.54   0.52      0.58          0.62      0.45   0.56   0.52      0.60          0.62      0.45   0.57   0.53      0.62          0.63      0.46
ridge_history            0.71   0.73      0.69          0.62      0.45   0.53   0.54      0.52          0.62      0.45   0.46   0.47      0.45          0.63      0.46
ridge_forecast_weather   0.75   0.88      0.66          0.62      0.45   0.73   0.86      0.63          0.62      0.45   0.71   0.86      0.60          0.63      0.46
ridge_forecast_no_wind   0.72   0.73      0.72          0.62      0.45   0.64   0.66      0.62          0.62      0.45   0.63   0.67      0.60          0.63      0.46
rf_history               0.70   0.76      0.66          0.62      0.45   0.58   0.60      0.57          0.62      0.45   0.58   0.64      0.53          0.63      0.46
rf_forecast_weather      0.78   0.92      0.67          0.62      0.45   0.72   0.91      0.60          0.62      0.45   0.70   0.89      0.57          0.63      0.46
rf_forecast_no_wind      0.74   0.76      0.72          0.62      0.45   0.65   0.66      0.63          0.62      0.45   0.59   0.59      0.58          0.63      0.46
xgb_history              0.63   0.58      0.70          0.62      0.45   0.47   0.43      0.51          0.62      0.45   0.44   0.38      0.55          0.63      0.46
xgb_forecast_weather     0.77   0.86      0.70          0.62      0.45   0.67   0.74      0.61          0.62      0.45   0.73   0.81      0.66          0.63      0.46
xgb_forecast_no_wind     0.73   0.71      0.76          0.62      0.45   0.62   0.55      0.69          0.62      0.45   0.64   0.56      0.75          0.63      0.46
lgbm_history             0.69   0.67      0.71          0.62      0.45   0.50   0.45      0.57          0.62      0.45   0.56   0.50      0.63          0.63      0.46
lgbm_forecast_weather    0.76   0.82      0.71          0.62      0.45   0.70   0.75      0.64          0.62      0.45   0.72   0.77      0.68          0.63      0.46
lgbm_forecast_no_wind    0.77   0.74      0.79          0.62      0.45   0.64   0.58      0.70          0.62      0.45   0.71   0.64      0.80          0.63      0.46

## Ablations

All use LightGBM on the forecast feature set. Weather on or off is read from the tables above (`*_history` against `*_forecast_weather`).

### Lag window (RQ3)

How many daily PM2.5 lags, and whether the rolling statistics are kept. The label `lagsN` means lags 0 to N-1 without rolling statistics.

In [10]:
abl, abl_fitted = {}, {}
for view in VIEWS:
    for lead in LEADS:
        models = ablations.lag_window_models()
        abl[(view, lead)] = run_models(daily, models, lead, blocks, view)
        abl_fitted[(view, lead)] = models
LAGS = [m.name for m in ablations.lag_window_models()]


def abl_table(store, view, names, group="winter"):
    cols = {}
    for lead in LEADS:
        s = report.summarize(common_rows(store[(view, lead)], names), names)
        s = s[s["group"] == group].set_index("model")
        cols[(f"lead {lead}", "MAE")] = s["mae"]
        cols[(f"lead {lead}", "n")] = s["n"]
    return pd.DataFrame(cols).round(1)


print("observed view, winter")
display(abl_table(abl, "observed", LAGS))
print("real forecast view, winter")
display(abl_table(abl, "forecast", LAGS))

observed view, winter


lead 1      lead 2      lead 3     
                      MAE    n    MAE    n    MAE    n
model                                                 
lgbm_lags1           15.6  467   18.3  456   19.1  446
lgbm_lags3           14.9  467   18.8  456   19.2  446
lgbm_lags7           15.2  467   18.7  456   19.3  446
lgbm_lags7_rolling   15.5  467   18.6  456   19.2  446

real forecast view, winter


lead 1      lead 2      lead 3     
                      MAE    n    MAE    n    MAE    n
model                                                 
lgbm_lags1           16.5  146   21.3  143   22.4  140
lgbm_lags3           16.8  146   21.3  143   23.1  140
lgbm_lags7           16.9  146   21.0  143   21.5  140
lgbm_lags7_rolling   17.1  146   20.9  143   22.0  140

In [11]:
rows = []
for view in VIEWS:
    row = {"view": view}
    for lead in LEADS:
        d = report.paired_difference(common_rows(abl[(view, lead)], LAGS), "lgbm_lags7_rolling", "lgbm_lags1", "winter")
        row[f"lead {lead}"] = f"{d['diff']:+.1f} [{d['lo']:+.1f}, {d['hi']:+.1f}]"
    rows.append(row)
print("lags7_rolling minus lags1 (winter MAE, negative favours the longer window)")
pd.DataFrame(rows).set_index("view")

lags7_rolling minus lags1 (winter MAE, negative favours the longer window)


,lead 1,lead 2,lead 3
view,,,
observed,"-0.1 [-0.5, +0.3]","+0.3 [-0.3, +1.0]","+0.0 [-0.5, +0.6]"
forecast,"+0.6 [-0.0, +1.3]","-0.4 [-1.6, +0.7]","-0.4 [-1.5, +0.8]"


### Wind bias correction

The forecast wind is biased low (see notebook 05). The correction multiplies forecast wind speed, gusts and wind components by the ratio observed/forecast, fitted only on training days that have a real archived forecast. It can only be tested on real forecasts, so only the forecast view is used.

In [12]:
wind, wind_fitted = {}, {}
for lead in LEADS:
    models = ablations.wind_bias_models()
    wind[("forecast", lead)] = run_models(daily, models, lead, blocks, "forecast")
    wind_fitted[("forecast", lead)] = models
WIND = [m.name for m in ablations.wind_bias_models()]
print("winter MAE, real forecasts")
display(abl_table(wind, "forecast", WIND))
print("all scored days")
display(abl_table(wind, "forecast", WIND, "all"))
rows = []
row = {}
for lead in LEADS:
    d = report.paired_difference(common_rows(wind[("forecast", lead)], WIND), "lgbm_wind_corrected", "lgbm_wind_raw", "winter")
    row[f"lead {lead}"] = f"{d['diff']:+.1f} [{d['lo']:+.1f}, {d['hi']:+.1f}]"
print("corrected minus raw (winter MAE, negative favours the correction)")
pd.DataFrame([row], index=["difference"])

winter MAE, real forecasts


lead 1      lead 2      lead 3     
                       MAE    n    MAE    n    MAE    n
model                                                  
lgbm_wind_raw         17.1  146   20.9  143   22.0  140
lgbm_wind_corrected   17.2  146   20.8  143   21.2  140

all scored days


lead 1      lead 2      lead 3     
                       MAE    n    MAE    n    MAE    n
model                                                  
lgbm_wind_raw         12.8  363   15.5  358   16.3  354
lgbm_wind_corrected   12.8  363   15.4  358   15.9  354

corrected minus raw (winter MAE, negative favours the correction)


,lead 1,lead 2,lead 3
difference,"+0.0 [-1.0, +0.8]","-0.1 [-1.8, +1.5]","-0.8 [-2.9, +1.3]"


In [13]:
print("wind ratio fitted in the last block, by lead:")
print({lead: wind_fitted[("forecast", lead)][1].wind_ratio for lead in LEADS})

wind ratio fitted in the last block, by lead:
{1: 1.5169560689289414, 2: 1.5187025657059214, 3: 1.5780569277273662}


In [14]:
if tracked_run is not None:
    n = log_runs(abl, abl_fitted, LAGS, "ablation-lags", "3.4.4 ablations")
    n += log_runs(wind, wind_fitted, WIND, "ablation-wind", "3.4.4 ablations")
    print("logged", n, "runs")

logged 30 runs


## Saved tables

In [15]:
out = []
for store, names in ((results, MAIN), (abl, LAGS), (wind, WIND)):
    for (view, lead), pred in store.items():
        s = report.summarize(common_rows(pred, names), names)
        s.insert(0, "lead", lead)
        s.insert(0, "view", view)
        out.append(s)
table = pd.concat(out, ignore_index=True)
target = ROOT / "reports" / "model_evaluation"
target.mkdir(parents=True, exist_ok=True)
table.round(4).to_csv(target / "model_comparison_metrics.csv", index=False)
print(f"saved reports/model_evaluation/model_comparison_metrics.csv ({len(table)} rows)")

saved reports/model_evaluation/model_comparison_metrics.csv (342 rows)
